# Weighted Alignment Scoring

## Metric: Per-Pair Weighted Alignment Score

For each camera pair, we measure how well the persona's **actual choice** aligns with their **stated priority ranking**, weighted so that higher-priority attributes count more.

### Inputs per pair
- `priority_order`: the persona's Turn-1 ranking of all 11 attributes (most → least important)
- `pair_attributes`: which attributes are visible for this pair (from the stimulus JSON)
- `camera_a`, `camera_b`: attribute values for each camera
- `choice`: `"A"` or `"B"` — which camera the persona chose
- `attribute_direction`: `"higher_is_better"` or `"lower_is_better"` per attribute

### Steps

**Step 1 — Filter to visible differing attributes.**  
Only consider attributes listed in `pair_attributes` where `camera_a[attr] != camera_b[attr]`. If both cameras have the same value on an attribute (e.g. both `weather_sealed = "Yes"`), that attribute is uninformative and excluded. Call the remaining set `D`.

**Step 2 — Determine which camera is "better" on each attribute in D.**  
Use `attribute_direction`: if `higher_is_better`, the camera with the higher value wins; if `lower_is_better`, the lower value wins. Boolean attributes (`Yes`/`No`) are converted to 1/0 before comparison; all four booleans are `higher_is_better`.

**Step 3 — Assign weight from the global priority rank.**  
Each attribute's weight comes from its position in the **full 11-attribute ranking** (not re-ranked within the pair). If `price_usd` is ranked #5 globally, it always gets the #5 weight — even in a simple pair where it might be the highest-ranked visible attribute. This keeps the ranking's meaning stable across all pairs.

**Step 4 — Compute the score.**

```
correct(attr) = 1  if the persona chose the camera that is better on attr
              = 0  otherwise

score = Σ weight(attr) × correct(attr)   /   Σ weight(attr)
        for attr in D                         for attr in D
```

**Range:** [0, 1]  
- 1.0 = chose the better camera on every differing attribute  
- 0.0 = chose the worse camera on every differing attribute  
- 0.5 = weighted wins and losses cancel out  

### Weight Schemes

We report results under multiple weight schemes to test robustness:

| Scheme | Formula (`rank` is 1-indexed) | Top:Bottom ratio | Character |
|---|---|---|---|
| **Linear** (primary) | `w = 12 - rank` | 11 : 1 | Moderate — every attribute has influence |
| **Reciprocal** | `w = 1 / rank` | 11 : 1 | Steeper initial drop, then flattens |
| **Geometric (α=0.5)** | `w = 0.5^(rank-1)` | 1024 : 1 | Heavy top-priority emphasis |
| **Lexicographic** | Only the top-ranked differing attr | binary | Original abstract metric |

### Exclusions
- **Empty choice** (parser failure): pair excluded, counted in exclusion report
- **No differing attributes** (`D` is empty): pair excluded (uninformative)

### Aggregation
- **Per-complexity alignment**: mean score across the 5 pairs at each complexity level
- **Per-session alignment**: mean score across all 15 pairs
- **Degradation slope**: per-session regression of score on complexity level (1, 2, 3); negative slope = coherence degrades with more attributes

### Statistical Model
- **Beta regression** (or linear mixed model on logit-transformed scores) with:
  - Fixed effects: `persona_label` (expert/novice) × `complexity` (simple/moderate/complex)
  - Random effects: `(1 | persona_id)`, `(1 | session_id)` nested within persona

In [ ]:
import json
import glob
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# ============================================================
# CONFIGURATION — change these paths if running outside Colab
# ============================================================
# If running locally, point PROJECT_ROOT to your repo clone.
# If running in Colab, this should match where you cloned the repo.
PROJECT_ROOT = Path(".")
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"

# Which result file(s) to score.
RESULT_FILES = [RESULTS_DIR / "llm_persona_outputs_merged.json"]

print(f"Result files: {[str(f) for f in RESULT_FILES]}")

In [ ]:
# ============================================================
# Load stimulus data (attributes + camera pairs)
# ============================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

attributes    = load_json(DATA_DIR / "attributes.json")
ATTR_POOL     = attributes["attribute_pool"]
ATTR_DESC     = attributes["attribute_descriptions"]
ATTR_DIR      = attributes["attribute_direction"]

STIMULI = {
    "simple":   load_json(DATA_DIR / "cameras_simple.json")["pairs"],
    "moderate": load_json(DATA_DIR / "cameras_moderate.json")["pairs"],
    "complex":  load_json(DATA_DIR / "cameras_complex.json")["pairs"],
}

# Build a lookup: (complexity, pair_id) -> pair dict
PAIR_LOOKUP = {}
for level, pairs in STIMULI.items():
    for pair in pairs:
        PAIR_LOOKUP[(level, str(pair["pair_id"]))] = pair

print(f"Attributes: {len(ATTR_POOL)}")
print(f"Pairs loaded: {len(PAIR_LOOKUP)} (5 simple + 5 moderate + 5 complex)")

In [ ]:
# ============================================================
# Load LLM session outputs
# ============================================================
sessions = []
seen = set()
for f in RESULT_FILES:
    with open(f, "r", encoding="utf-8") as fp:
        data = json.load(fp)
    for s in data:
        key = (s["persona_id"], s["generation_id"])
        if key in seen:
            continue  # skip duplicates
        seen.add(key)
        sessions.append(s)

print(f"Loaded {len(sessions)} sessions from {len(RESULT_FILES)} file(s)")
print(f"Personas represented: {sorted(set(s['persona_id'] for s in sessions))}")

# Quick data-quality report
for s in sessions:
    pid = s["persona_id"]
    n_ranked = len(set(s["priority_order"]) & set(ATTR_POOL))
    empty = sum(
        1 for lvl in ["simple", "moderate", "complex"]
        for p in s.get(lvl, {}).values()
        if not p.get("choice")
    )
    total = sum(len(s.get(lvl, {})) for lvl in ["simple", "moderate", "complex"])
    flag = ""
    if n_ranked < 11:
        flag += f" [WARN: only {n_ranked}/11 attrs in ranking]"
    if empty > 0:
        flag += f" [WARN: {empty}/{total} empty choices]"
    print(f"  {pid} gen={s['generation_id']} label={s['persona_label']} "
          f"ranked={n_ranked}/11 empty={empty}/{total}{flag}")

In [ ]:
# ============================================================
# Weight functions
# ============================================================
# Each takes a 1-indexed rank (1 = most important) and returns a weight.
# Higher weight = more influence on the alignment score.

N_ATTRS = len(ATTR_POOL)  # 11

def weight_linear(rank):
    """w = N - rank + 1. Top=11, Bottom=1."""
    return N_ATTRS - rank + 1

def weight_reciprocal(rank):
    """w = 1/rank. Top=1.0, Bottom=0.091."""
    return 1.0 / rank

def weight_geometric(rank, alpha=0.5):
    """w = alpha^(rank-1). Top=1.0, Bottom=alpha^10."""
    return alpha ** (rank - 1)


# All schemes to evaluate (name -> function)
WEIGHT_SCHEMES = {
    "linear":      weight_linear,
    "reciprocal":  weight_reciprocal,
    "geometric":   lambda r: weight_geometric(r, alpha=0.5),
}

# Preview weights for each scheme
print(f"{'Rank':<6}", end="")
for name in WEIGHT_SCHEMES:
    print(f"{name:>14}", end="")
print()
for rank in range(1, N_ATTRS + 1):
    print(f"{rank:<6}", end="")
    for name, fn in WEIGHT_SCHEMES.items():
        print(f"{fn(rank):>14.4f}", end="")
    print()

In [ ]:
# ============================================================
# Core scoring function
# ============================================================

def to_numeric(val):
    """Convert boolean string to int for comparison."""
    if isinstance(val, str):
        return 1 if val.strip().lower() == "yes" else 0
    return val


def score_pair_weighted(choice, pair, priority_order, weight_fn):
    """Compute the weighted alignment score for one pair.

    Args:
        choice: "A" or "B"
        pair: stimulus dict with camera_a, camera_b, pair_attributes
        priority_order: list of 11 attribute keys, most → least important
        weight_fn: callable(rank) -> float, rank is 1-indexed

    Returns:
        float in [0, 1], or None if unscorable (empty choice / no differing attrs)
    """
    if not choice or choice not in ("A", "B"):
        return None

    rank_of = {attr: i + 1 for i, attr in enumerate(priority_order)}
    cam_a = pair["camera_a"]
    cam_b = pair["camera_b"]

    numerator = 0.0
    denominator = 0.0

    for attr in pair["pair_attributes"]:
        val_a = to_numeric(cam_a[attr])
        val_b = to_numeric(cam_b[attr])

        # Skip attributes where cameras are equal — no tradeoff to test
        if val_a == val_b:
            continue

        # Which camera is better on this attribute?
        direction = ATTR_DIR[attr]
        if direction == "higher_is_better":
            winner = "A" if val_a > val_b else "B"
        else:  # lower_is_better
            winner = "A" if val_a < val_b else "B"

        rank = rank_of.get(attr, N_ATTRS)  # fallback to lowest if missing
        w = weight_fn(rank)
        correct = 1.0 if choice == winner else 0.0

        numerator += w * correct
        denominator += w

    if denominator == 0:
        return None  # no differing attributes

    return numerator / denominator


def score_pair_lexicographic(choice, pair, priority_order):
    """Binary lexicographic alignment: does the choice match the prediction
    based on the highest-ranked differing attribute?

    Returns: 1.0, 0.0, or None if unscorable.
    """
    if not choice or choice not in ("A", "B"):
        return None

    rank_of = {attr: i + 1 for i, attr in enumerate(priority_order)}
    cam_a = pair["camera_a"]
    cam_b = pair["camera_b"]

    # Find all differing attributes, sorted by rank (highest priority first)
    differing = []
    for attr in pair["pair_attributes"]:
        val_a = to_numeric(cam_a[attr])
        val_b = to_numeric(cam_b[attr])
        if val_a != val_b:
            differing.append((rank_of.get(attr, N_ATTRS), attr))

    if not differing:
        return None

    differing.sort()  # sort by rank (lowest number = highest priority)
    top_rank, top_attr = differing[0]

    val_a = to_numeric(cam_a[top_attr])
    val_b = to_numeric(cam_b[top_attr])
    direction = ATTR_DIR[top_attr]

    if direction == "higher_is_better":
        predicted = "A" if val_a > val_b else "B"
    else:
        predicted = "A" if val_a < val_b else "B"

    return 1.0 if choice == predicted else 0.0


print("Scoring functions defined.")

In [ ]:
# ============================================================
# Score all sessions → long-form DataFrame
# ============================================================
# One row per (session, pair). Columns include the score under
# every weight scheme plus lexicographic, so downstream analysis
# can filter/pivot without re-scoring.

rows = []

for session in sessions:
    pid   = session["persona_id"]
    pname = session["persona_name"]
    label = session["persona_label"]
    gen   = session["generation_id"]
    porder = session["priority_order"]

    for level in ["simple", "moderate", "complex"]:
        for pair_id_str, result in session.get(level, {}).items():
            choice = result.get("choice", "")

            pair_key = (level, pair_id_str)
            if pair_key not in PAIR_LOOKUP:
                print(f"  WARNING: pair {pair_key} not found in stimulus data, skipping")
                continue
            pair = PAIR_LOOKUP[pair_key]

            row = {
                "persona_id":    pid,
                "persona_name":  pname,
                "persona_label": label,
                "generation_id": gen,
                "complexity":    level,
                "pair_id":       int(pair_id_str),
                "choice":        choice,
                "choice_valid":  choice in ("A", "B"),
            }

            # Score under each weighted scheme
            for scheme_name, weight_fn in WEIGHT_SCHEMES.items():
                row[f"score_{scheme_name}"] = score_pair_weighted(
                    choice, pair, porder, weight_fn
                )

            # Lexicographic (binary)
            row["score_lexicographic"] = score_pair_lexicographic(
                choice, pair, porder
            )

            rows.append(row)

df = pd.DataFrame(rows)

# Add a numeric complexity code for regression
complexity_code = {"simple": 1, "moderate": 2, "complex": 3}
df["complexity_code"] = df["complexity"].map(complexity_code)

print(f"Scored {len(df)} pair trials across {df['persona_id'].nunique()} personas")
print(f"Empty choices excluded: {(~df['choice_valid']).sum()} / {len(df)}")
print()
df.head(10)

In [ ]:
# ============================================================
# Summary: mean alignment by persona_label × complexity
# ============================================================
# Filter to valid choices only
valid = df[df["choice_valid"]].copy()

score_cols = [c for c in df.columns if c.startswith("score_")]

summary = (
    valid
    .groupby(["persona_label", "complexity"])[score_cols]
    .agg(["mean", "std", "count"])
)

# Flatten multi-level columns for readability
summary.columns = [f"{col[0]}_{col[1]}" for col in summary.columns]
print(summary.to_string())

In [ ]:
# ============================================================
# Per-persona alignment (primary weight scheme: linear)
# ============================================================
per_persona = (
    valid
    .groupby(["persona_id", "persona_label", "persona_name"])
    .agg(
        mean_linear=  ("score_linear", "mean"),
        std_linear=   ("score_linear", "std"),
        mean_lexico=  ("score_lexicographic", "mean"),
        n_pairs=      ("score_linear", "count"),
    )
    .sort_values("mean_linear", ascending=False)
)

print(per_persona.to_string())

In [ ]:
# ============================================================
# Per-complexity alignment (primary weight scheme: linear)
# ============================================================
per_complexity = (
    valid
    .groupby(["persona_label", "complexity", "complexity_code"])
    .agg(
        mean_score=("score_linear", "mean"),
        std_score= ("score_linear", "std"),
        n_pairs=   ("score_linear", "count"),
    )
    .sort_values(["persona_label", "complexity_code"])
)

print("Mean weighted alignment (linear) by label x complexity:")
print(per_complexity.to_string())
print()

# Degradation: is there a negative trend as complexity increases?
for label in ["expert", "novice"]:
    subset = valid[valid["persona_label"] == label]
    if len(subset) < 3:
        continue
    corr = subset[["complexity_code", "score_linear"]].corr().iloc[0, 1]
    print(f"{label}: correlation(complexity, score) = {corr:.3f} "
          f"({'degradation' if corr < 0 else 'no degradation'})")

In [ ]:
# ============================================================
# Robustness check: do findings hold across weight schemes?
# ============================================================
print("Mean alignment by persona_label across all weight schemes:\n")

for scheme in score_cols:
    scheme_name = scheme.replace("score_", "")
    by_label = valid.groupby("persona_label")[scheme].mean()
    expert_mean = by_label.get("expert", float("nan"))
    novice_mean = by_label.get("novice", float("nan"))
    gap = expert_mean - novice_mean
    print(f"  {scheme_name:15s}  expert={expert_mean:.3f}  novice={novice_mean:.3f}  gap={gap:+.3f}")

In [ ]:
# ============================================================
# Export scored data for external analysis (R, etc.)
# ============================================================
output_csv = RESULTS_DIR / "scored_pair_trials.csv"
df.to_csv(output_csv, index=False)
print(f"Exported {len(df)} rows to {output_csv}")
print(f"Columns: {list(df.columns)}")